# DarkSpot - Анализ коммерческой аренды (ЦИАН)

**DarkSpot** - data-driven платформа для выбора лучших локаций под запуск офлайн-бизнеса.  
Этот блок отвечает за сбор и анализ данных о коммерческой аренде с ЦИАН.

## О блоке

Источник данных: **ЦИАН** (cian.ru) - крупнейший российский агрегатор объявлений о недвижимости.  
Тип парсинга: **статический** (cloudscraper + BeautifulSoup).  
Регион: **Москва**.

## Что анализируем

| Вопрос | Метрика |
|---|---|
| Сколько стоит аренда по типам помещений? | `price_sqm_month` - руб/м²/мес |
| Как цена зависит от расстояния до метро? | `metro_walk_min` - минуты пешком |
| Какова структура предложения по округам? | `okrug`, `district` |
| Насколько помещения оборудованы? | `renovation`, `has_furniture`, `building_class` |


## Методология

### Источник данных

ЦИАН использует динамический рендеринг (JavaScript), однако начальные данные страницы передаются  
через server-side rendering в виде JSON-объекта внутри script-тега (`window._cianConfig`).  
Это позволяет извлекать структурированные данные без полноценного выполнения JavaScript -  
с помощью HTTP-запроса и парсинга HTML.

**Инструменты:**
- `cloudscraper` - обход Cloudflare WAF и получение HTML страницы
- `BeautifulSoup` - парсинг HTML и извлечение script-тега с данными
- `json` - разбор встроенного JSON с массивом офферов
- `ThreadPoolExecutor` - параллельный сбор по типам объектов

### Типы объектов

| Тип | URL ЦИАН |
|---|---|
| Офис | `snyat-ofis` |
| Торговая площадь | `snyat-torgovuyu-ploshad` |
| Свободное назначение | `snyat-pomeshenie-svobodnogo-naznachenija` |
| Склад | `snyat-sklad` |

### Поля датасета

| Поле | Описание |
|---|---|
| `price_month` | Итоговая цена аренды в месяц, руб |
| `price_sqm_month` | Цена за м² в месяц, руб/м² |
| `area_sqm` | Площадь помещения, м² |
| `okrug` / `district` | Административный округ и район Москвы |
| `metro_station` | Ближайшая станция метро (пешком) |
| `metro_walk_min` | Расстояние до метро пешком, мин |
| `floor` / `floors_total` | Этаж и этажность здания |
| `building_class` | Класс здания (A, B+, B, C) |
| `building_type` | Тип здания (БЦ, ТЦ, жилой дом и др.) |
| `renovation` | Вид ремонта |
| `ceiling_height_m` | Высота потолков из описания, м |


---
# Часть 1. Окружение

In [5]:
!pip install -q playwright plotly dash pandas numpy
!playwright install chromium --with-deps
!pip install -q cloudscraper beautifulsoup4
!pip install -q jupyter-dash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 121.3 MB/s eta 0:00:00
Installing dependencies...
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cli.github.com/packages stable InRelease [3,917 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,644 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu

In [6]:
import os, re, json, time, random, threading, warnings
from abc import ABC, abstractmethod
from typing import List, Dict, Optional
import cloudscraper
from bs4 import BeautifulSoup

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)

---
# Часть 2. ООП-архитектура

Базовый класс задаёт интерфейс и кеширование. Конкретный парсер реализует `fetch()` и `parse()`.

In [7]:
class BaseParser(ABC):
    """Абстрактный базовый класс для всех парсеров DarkSpot."""

    def __init__(self, cache_dir: str = "darkspot_cache", use_cache: bool = True):
        self.cache_dir   = cache_dir
        self.use_cache   = use_cache
        self.df          = None
        self.source_name = self.__class__.__name__

    @property
    def cache_path(self) -> str:
        return os.path.join(self.cache_dir, f"{self.source_name}_raw.csv")

    @abstractmethod
    def fetch(self) -> List[Dict]: ...

    @abstractmethod
    def parse(self, raw: List[Dict]) -> pd.DataFrame: ...

    def collect(self) -> pd.DataFrame:
        """fetch → сохранить сырые данные в CSV → вернуть DataFrame."""
        if self.use_cache and os.path.exists(self.cache_path):
            print(f"[{self.source_name}] Кэш найден: {self.cache_path}")
            self.df = pd.read_csv(self.cache_path)
            return self.df

        print(f"🔍 [{self.source_name}] Запускаем парсинг...")
        raw     = self.fetch()
        self.df = self.parse(raw)

        os.makedirs(self.cache_dir, exist_ok=True)
        self.df.to_csv(self.cache_path, index=False)
        print(f"Сырые данные → {self.cache_path}  ({len(self.df)} строк)")
        return self.df


print("Все норм")

Все норм


---
# Часть 3. CianCommercialParser

**Тип парсинга:** статический - cloudscraper + BeautifulSoup.

ЦИАН использует Cloudflare WAF и JavaScript-рендеринг. `cloudscraper` обходит защиту на уровне  
HTTP-заголовков, имитируя браузер Chrome. Данные извлекаются из script-тега с JSON внутри HTML.

**Многопоточность:** типы объектов (офисы, склады, торговые площади, свободное назначение)  
парсятся параллельно через `ThreadPoolExecutor`, что ускоряет сбор в 3-4 раза.


In [8]:
class CianCommercialParser(BaseParser):
    """
    Парсер коммерческой аренды ЦИАН.
    Тип парсинга: статический (cloudscraper + BeautifulSoup).
    Поддержка многопоточности: типы объектов парсятся параллельно.
    """

    OFFER_TYPE_IDS = {
        "offices":               "1",
        "shoppingAreas":         "2",
        "warehouse":             "3",
        "freeAppointmentObject": "5",
    }

    OFFER_TYPE_NAMES = {
        "offices":               "Офис",
        "shoppingAreas":         "Торговая площадь",
        "warehouse":             "Склад",
        "freeAppointmentObject": "Свободное назначение",
    }

    DEC_MAP = {
        "design":   "дизайнерский",
        "euro":     "евроремонт",
        "fine":     "евроремонт",
        "cosmetic": "косметический",
        "no":       "без отделки",
    }

    CLS_MAP = {
        "a": "A", "b_plus": "B+", "bPlus": "B+",
        "b": "B", "c": "C", "d": "C",
    }

    BUILDING_TYPE_MAP = {
        "businessCenter":         "Бизнес-центр",
        "shoppingCenter":         "Торговый центр",
        "shoppingEntertainment":  "ТРЦ",
        "warehouse":              "Склад",
        "manufacturingFacility":  "Производство",
        "residentialHouse":       "Жилой дом",
        "standalone":             "Отдельное здание",
        "administrativeBuilding": "Административное здание",
    }

    def __init__(self, offer_types=None, max_pages=5, max_workers=4, **kwargs):
        super().__init__(**kwargs)
        self.offer_types = offer_types or ["shoppingAreas"]
        self.max_pages   = max_pages
        self.max_workers = max_workers
        self._lock       = __import__("threading").Lock()

    def _make_scraper(self):
        return cloudscraper.create_scraper(
            browser={"browser": "chrome", "platform": "darwin", "mobile": False}
        )

    def fetch(self) -> List[Dict]:
        from concurrent.futures import ThreadPoolExecutor, as_completed

        all_raw = []
        futures = {}

        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            for offer_type in self.offer_types:
                future = executor.submit(self._fetch_type, offer_type)
                futures[future] = offer_type

            for future in as_completed(futures):
                offer_type = futures[future]
                try:
                    rows = future.result()
                    all_raw.extend(rows)
                except Exception as e:
                    print(f"  ✗ [{self.OFFER_TYPE_NAMES.get(offer_type)}]: {e}")

        return all_raw

    def _fetch_type(self, offer_type: str) -> List[Dict]:
        scraper      = self._make_scraper()
        type_id      = self.OFFER_TYPE_IDS.get(offer_type, "2")
        name         = self.OFFER_TYPE_NAMES.get(offer_type, offer_type)
        type_raw     = []

        for p_num in range(1, self.max_pages + 1):
            url = (
                f"https://www.cian.ru/cat.php?deal_type=rent&engine_version=2"
                f"&offer_type=offices&office_type%5B0%5D={type_id}&p={p_num}&region=1"
            )
            try:
                resp = scraper.get(url, timeout=20)

                if resp.status_code == 403:
                    with self._lock:
                        print(f"  ⛔ [{name}] стр.{p_num}: 403")
                    break
                if resp.status_code != 200:
                    with self._lock:
                        print(f"  ⚠️  [{name}] стр.{p_num}: {resp.status_code}")
                    break

                rows = self._extract(resp.text, offer_type)

                if not rows:
                    with self._lock:
                        print(f"  ⚠️  [{name}] стр.{p_num}: данные не найдены")
                    break

                type_raw.extend(rows)
                with self._lock:
                    print(f"  ✓ [{name}] стр.{p_num}: +{len(rows)} (итого {len(type_raw)})")

                if p_num % 10 == 0:
                    time.sleep(random.uniform(8.0, 12.0))
                else:
                    time.sleep(random.uniform(0.7, 1.2))

            except Exception as e:
                with self._lock:
                    print(f"  ✗ [{name}] стр.{p_num}: {e}")
                break

        return type_raw

    def _extract(self, html: str, offer_type: str) -> List[Dict]:
        soup = BeautifulSoup(html, "html.parser")

        script_text = None
        for script in soup.find_all("script"):
            t = script.string or ""
            if "bargainTerms" in t and len(t) > 100_000:
                script_text = t
                break

        if not script_text:
            return []

        key = '"offers":'
        idx = script_text.find(key)
        while idx >= 0:
            arr_start = script_text.index("[", idx)
            depth, i  = 0, arr_start
            while i < len(script_text):
                if   script_text[i] == "[": depth += 1
                elif script_text[i] == "]":
                    depth -= 1
                    if depth == 0:
                        candidate = script_text[arr_start: i + 1]
                        if '"bargainTerms"' in candidate:
                            try:
                                offers = json.loads(candidate)
                                return [r for o in offers if o
                                        for r in [self._row(o, offer_type)] if r]
                            except Exception:
                                pass
                        break
                i += 1
            idx = script_text.find(key, idx + 1)

        return []

    def _row(self, o: dict, offer_type: str) -> Optional[Dict]:
        try:
            bt        = o.get("bargainTerms", {}) or {}
            price_rur = float(
                o.get("priceTotalPerMonthRur") or
                bt.get("priceRur") or
                bt.get("price") or
                0
            )
            area = float(o.get("totalArea") or o.get("minArea") or 0)
            if price_rur <= 0 or area <= 0:
                return None

            price_sqm = round(price_rur / area, 1)
            if price_sqm < 100:
                return None

            geo = o.get("geo", {}) or {}
            district = okrug = street = house = ""
            for a in geo.get("address", []) or []:
                t, n = a.get("type", ""), a.get("name", "")
                if t == "raion":    district = n
                elif t == "okrug":  okrug    = n
                elif t == "street": street   = n
                elif t == "house":  house    = n

            address = ", ".join(filter(None, [street, house]))

            ug          = geo.get("undergrounds") or []
            walk_metros = [u for u in ug if u.get("transportType") == "walk"]
            metro       = walk_metros[0].get("name") if walk_metros else None
            metro_min   = walk_metros[0].get("time") if walk_metros else None

            bld      = o.get("building", {}) or {}
            bld_cls  = self.CLS_MAP.get((bld.get("classType") or "").lower(), None)
            bld_type = self.BUILDING_TYPE_MAP.get(bld.get("type") or "", None)

            renovation = self.DEC_MAP.get(o.get("decoration") or "", None)

            floor_raw   = o.get("floorNumber")
            floor       = int(floor_raw)   if floor_raw   is not None else None
            f_total_raw = bld.get("floorsCount")
            f_total     = int(f_total_raw) if f_total_raw is not None else None

            desc       = o.get("description", "") or ""
            ceil_match = re.search(r'высота потолков[:\s]+(\d+[.,]\d+)', desc, re.IGNORECASE)
            ceiling    = float(ceil_match.group(1).replace(",", ".")) if ceil_match else None

            return {
                "cian_id":          int(o.get("id", 0)),
                "url":              f"https://www.cian.ru/rent/commercial/{o.get('id', 0)}/",
                "object_type":      self.OFFER_TYPE_NAMES.get(offer_type, offer_type),
                "price_month":      int(price_rur),
                "area_sqm":         area,
                "price_sqm_month":  price_sqm,
                "address":          address or None,
                "district":         district or None,
                "okrug":            okrug or None,
                "metro_station":    metro,
                "metro_walk_min":   metro_min,
                "floor":            floor,
                "floors_total":     f_total,
                "building_class":   bld_cls,
                "building_type":    bld_type,
                "renovation":       renovation,
                "has_furniture":    o.get("hasFurniture"),
                "has_ac":           None,
                "has_ventilation":  None,
                "ceiling_height_m": ceiling,
                "photos_count":     len(o.get("photos") or []),
                "source":           "cloudscraper",
            }
        except Exception:
            return None

    def parse(self, raw: List[Dict]) -> pd.DataFrame:
        if not raw:
            raise ValueError("Нет данных")
        df = pd.DataFrame(raw)
        df = df[(df["price_month"] > 0) & (df["area_sqm"] > 0)]
        return df.reset_index(drop=True)


print("✅ CianCommercialParser определён")

✅ CianCommercialParser определён


---
# Часть 4. Сбор сырых данных

Запускаем парсер. Результат сохраняется в `darkspot_cache/CianCommercialParser_raw.csv`.  


In [9]:
parser = CianCommercialParser(
    offer_types=["shoppingAreas", "offices", "freeAppointmentObject", "warehouse"],
    max_pages=5,
    cache_dir="darkspot_cache",
    use_cache=False,
)

df_raw = parser.collect()

print(f"\n{'─'*55}")
print(f"  Сырых записей:  {len(df_raw):,}")
print(f"  Типов объектов: {df_raw['object_type'].value_counts().to_dict()}")
print(f"{'─'*55}")
df_raw.head(10)

🔍 [CianCommercialParser] Запускаем парсинг...
  ✓ [Склад] стр.1: +28 (итого 28)
  ✓ [Свободное назначение] стр.1: +28 (итого 28)
  ✓ [Офис] стр.1: +28 (итого 28)
  ✓ [Торговая площадь] стр.1: +28 (итого 28)
  ✓ [Склад] стр.2: +28 (итого 56)
  ✓ [Офис] стр.2: +28 (итого 56)
  ✓ [Свободное назначение] стр.2: +28 (итого 56)
  ✓ [Торговая площадь] стр.2: +28 (итого 56)
  ✓ [Склад] стр.3: +28 (итого 84)
  ✓ [Свободное назначение] стр.3: +28 (итого 84)
  ✓ [Офис] стр.3: +28 (итого 84)
  ✓ [Торговая площадь] стр.3: +28 (итого 84)
  ✓ [Склад] стр.4: +28 (итого 112)
  ✓ [Свободное назначение] стр.4: +28 (итого 112)
  ✓ [Торговая площадь] стр.4: +28 (итого 112)
  ✓ [Офис] стр.4: +28 (итого 112)
  ✓ [Склад] стр.5: +28 (итого 140)
  ✓ [Свободное назначение] стр.5: +28 (итого 140)
  ✓ [Торговая площадь] стр.5: +28 (итого 140)
  ✓ [Офис] стр.5: +28 (итого 140)
Сырые данные → darkspot_cache/CianCommercialParser_raw.csv  (560 строк)

───────────────────────────────────────────────────────
  Сырых запи

,cian_id,url,object_type,price_month,area_sqm,price_sqm_month,address,district,okrug,metro_station,metro_walk_min,floor,floors_total,building_class,building_type,renovation,has_furniture,has_ac,has_ventilation,ceiling_height_m,photos_count,source
0,328854598,https://www.cian.ru/rent/commercial/328854598/,Склад,10448250,9557.0,1093.3,1с2,None,ТАО (Троицкий),None,NaN,1,1,None,None,None,None,None,None,NaN,14,cloudscraper
1,328852638,https://www.cian.ru/rent/commercial/328852638/,Склад,2382142,2041.0,1167.1,"Евсеевская, 15",None,НАО (Новомосковский),None,NaN,1,1,None,None,None,None,None,None,NaN,14,cloudscraper
2,322491104,https://www.cian.ru/rent/commercial/322491104/,Склад,378015,213.9,1767.3,"2-й Павелецкий, 5С1",Даниловский,ЮАО,None,NaN,3,8,B,Бизнес-центр,None,None,None,None,NaN,16,cloudscraper
3,328854031,https://www.cian.ru/rent/commercial/328854031/,Склад,2170667,2024.0,1072.5,82,None,ТАО (Троицкий),None,NaN,1,1,None,None,None,None,None,None,NaN,15,cloudscraper
4,326542571,https://www.cian.ru/rent/commercial/326542571/,Склад,273217,252.2,1083.3,"Щелковское, 3с18",Гольяново,ВАО,Черкизовская,6.0,1,1,None,None,None,None,None,None,NaN,14,cloudscraper
5,327530388,https://www.cian.ru/rent/commercial/327530388/,Склад,1320000,1200.0,1100.0,"Конструктора Гуськова, 14с1",Матушкино,ЗелАО,None,NaN,2,2,None,None,None,None,None,None,NaN,7,cloudscraper
6,295111990,https://www.cian.ru/rent/commercial/295111990/,Склад,197750,85.9,2302.1,"Вернадского, 105",Тропарево-Никулино,ЗАО,Юго-Западная,5.0,-1,3,None,Торговый центр,None,None,None,None,NaN,23,cloudscraper
7,328853234,https://www.cian.ru/rent/commercial/328853234/,Склад,4741092,4063.0,1166.9,None,None,НАО (Новомосковский),None,NaN,1,1,None,None,None,None,None,None,NaN,10,cloudscraper
8,324606436,https://www.cian.ru/rent/commercial/324606436/,Склад,3300000,3000.0,1100.0,"Конструктора Гуськова, 14с2",Матушкино,ЗелАО,None,NaN,1,3,None,None,None,None,None,None,4.5,10,cloudscraper
9,328707595,https://www.cian.ru/rent/commercial/328707595/,Склад,388363,235.8,1647.0,"4922-й, 4с3",Старое Крюково,ЗелАО,None,NaN,3,4,None,None,None,None,None,None,NaN,10,cloudscraper


In [10]:
# Сводка по сырым данным
print("Типы и диапазоны значений:")
df_raw.info()
print()
print(df_raw[["price_month","area_sqm","price_sqm_month","metro_walk_min"]].describe().round(1))

Типы и диапазоны значений:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 560 entries, 0 to 559
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   cian_id           560 non-null    int64  
 1   url               560 non-null    object 
 2   object_type       560 non-null    object 
 3   price_month       560 non-null    int64  
 4   area_sqm          560 non-null    float64
 5   price_sqm_month   560 non-null    float64
 6   address           548 non-null    object 
 7   district          516 non-null    object 
 8   okrug             560 non-null    object 
 9   metro_station     468 non-null    object 
 10  metro_walk_min    468 non-null    float64
 11  floor             560 non-null    int64  
 12  floors_total      560 non-null    int64  
 13  building_class    158 non-null    object 
 14  building_type     182 non-null    object 
 15  renovation        0 non-null      object 
 16  has_furniture    

---
# Часть 5. Обработка данных

Чистим сырые данные: убираем дубли, выбросы, добавляем производные признаки.

In [11]:
df = df_raw.copy()

# ── 1. Дедупликация ───────────────────────────────────────────────────────────
print(f"Исходных записей: {len(df):,}")

# По cian_id
df_id  = df[df["cian_id"] > 0].drop_duplicates(subset=["cian_id"], keep="first")
df_nid = df[df["cian_id"] == 0].drop_duplicates(subset=["price_month", "area_sqm"], keep="first")
df = pd.concat([df_id, df_nid], ignore_index=True)
print(f"После дедупликации по ID: {len(df):,}")

# Полные дубли — совпадение по всем полям кроме технических
ignore_cols = ["cian_id", "url", "source"]
content_cols = [c for c in df.columns if c not in ignore_cols]
df = df.drop_duplicates(subset=content_cols, keep="first")
print(f"После дедупликации полных дублей: {len(df):,}")

# ── 2. Фильтрация выбросов ────────────────────────────────────────────────────
q_low  = df["price_sqm_month"].quantile(0.01)
q_high = df["price_sqm_month"].quantile(0.99)
df = df[df["price_sqm_month"].between(q_low, q_high)]
print(f"После фильтрации выбросов: {len(df):,}")
print(f"  Диапазон цен за м²: {q_low:,.0f} – {q_high:,.0f} руб/м²/мес")

# ── 3. Производные признаки ───────────────────────────────────────────────────
df["price_tier_per_meter"] = pd.cut(
    df["price_sqm_month"],
    bins=[0, 5_000, 15_000, 40_000, 999_999],
    labels=["эконом (<5k)", "средний (5–15k)", "премиум (15–40k)", "люкс (>40k)"]
)

df["area_tier"] = pd.cut(
    df["area_sqm"],
    bins=[0, 30, 80, 200, 9999],
    labels=["малый (<30 м²)", "средний (30–80)", "большой (80–200)", "крупный (>200)"]
)

df["metro_tier"] = pd.cut(
    df["metro_walk_min"],
    bins=[-1, 5, 10, 20, 999],
    labels=["≤5 мин", "6–10 мин", "11–20 мин", ">20 мин"]
)

df = df.reset_index(drop=True)

print(f"\n✅ Финальный датасет: {len(df):,} записей × {len(df.columns)} признаков")
print(df[["object_type", "price_month", "area_sqm", "price_sqm_month",
          "price_tier_per_meter", "area_tier", "metro_tier"]].head(8))

Исходных записей: 560
После дедупликации по ID: 454
После дедупликации полных дублей: 453
После фильтрации выбросов: 443
  Диапазон цен за м²: 908 – 38,173 руб/м²/мес

✅ Финальный датасет: 443 записей × 25 признаков
  object_type  price_month  area_sqm  price_sqm_month price_tier_per_meter  \
0       Склад     10448250    9557.0           1093.3         эконом (<5k)   
1       Склад      2382142    2041.0           1167.1         эконом (<5k)   
2       Склад       378015     213.9           1767.3         эконом (<5k)   
3       Склад      2170667    2024.0           1072.5         эконом (<5k)   
4       Склад       273217     252.2           1083.3         эконом (<5k)   
5       Склад      1320000    1200.0           1100.0         эконом (<5k)   
6       Склад       197750      85.9           2302.1         эконом (<5k)   
7       Склад      4741092    4063.0           1166.9         эконом (<5k)   

          area_tier metro_tier  
0    крупный (>200)        NaN  
1    крупный (>

In [12]:
# Итоговый DataFrame
print(f"Финальный датасет: {len(df)} записей × {len(df.columns)} признаков")
print()
print("Распределение по ценовым категориям:")
print(df["price_tier_per_meter"].value_counts().sort_index())
print()
print("Распределение по площади:")
print(df["area_tier"].value_counts().sort_index())

Финальный датасет: 443 записей × 25 признаков

Распределение по ценовым категориям:
price_tier_per_meter
эконом (<5k)        341
средний (5–15k)      90
премиум (15–40k)     12
люкс (>40k)           0
Name: count, dtype: int64

Распределение по площади:
area_tier
малый (<30 м²)       62
средний (30–80)     107
большой (80–200)    131
крупный (>200)      142
Name: count, dtype: int64


---
# Часть 6. Анализ данных

Анализируем собранные данные по четырём направлениям:

1. **Распределение цен** - как распределены цены аренды, где медиана рынка
2. **Цена vs площадь** - зависимость ставки от размера помещения
3. **Влияние метро** - как близость к метро влияет на цену аренды
4. **Структура рынка** - соотношение ценовых сегментов


## 6.1. Распределение цен

In [13]:
fig = px.histogram(
    df, x="price_sqm_month",
    nbins=40,
    title="Распределение цены аренды торговых площадей (руб/м²/мес)",
    labels={"price_sqm_month": "Цена, руб/м²/мес", "count": "Объявлений"},
    color_discrete_sequence=["#1d4ed8"],
)
fig.add_vline(x=df["price_sqm_month"].median(), line_dash="dash", line_color="red",
              annotation_text=f"Медиана: {df['price_sqm_month'].median():,.0f}",
              annotation_position="top right")
fig.update_layout(plot_bgcolor="white", height=400)
fig.show()

## 6.2. Цена vs площадь

In [14]:
fig = px.scatter(
    df,
    x="area_sqm", y="price_sqm_month",
    size="price_month", size_max=20,
    color="price_tier_per_meter",
    hover_data=["metro_station","metro_walk_min","floor"],
    title="Площадь vs Цена за м² (размер точки = общая цена аренды)",
    labels={"area_sqm": "Площадь, м²", "price_sqm_month": "Цена, руб/м²/мес",
            "price_tier_per_meter": "Категория"},
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_layout(plot_bgcolor="white", height=460)
fig.show()

## 6.3. Цена vs расстояние до метро

In [15]:
metro_stats = (
    df[df["metro_walk_min"] > 0]
    .groupby("metro_tier", observed=True)["price_sqm_month"]
    .agg(["median","count"])
    .reset_index()
    .rename(columns={"median":"Медиана цены","count":"Объявлений"})
)

fig = px.bar(
    metro_stats,
    x="metro_tier", y="Медиана цены",
    text="Медиана цены",
    title="Медианная цена аренды vs удалённость от метро",
    labels={"metro_tier": "Расстояние до метро", "Медиана цены": "руб/м²/мес"},
    color="Медиана цены",
    color_continuous_scale="Blues",
)
fig.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
fig.update_layout(plot_bgcolor="white", height=390, coloraxis_showscale=False)
fig.show()

## 6.4. Распределение по ценовым категориям

In [16]:
tier_stats = df["price_tier_per_meter"].value_counts().sort_index().reset_index()
tier_stats.columns = ["Категория","Кол-во"]

fig = px.pie(
    tier_stats,
    names="Категория", values="Кол-во",
    title="Структура рынка торговой аренды по ценовым категориям",
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Pastel,
)
fig.update_layout(height=400)
fig.show()

## 6.5. Топ-10 объявлений по соотношению цена/площадь

In [17]:
top = (
    df.sort_values("price_sqm_month")
    .head(10)
    [["url","price_month","area_sqm","price_sqm_month",
      "metro_station","metro_walk_min","floor","floors_total"]]
    .reset_index(drop=True)
)
top.index += 1
display(top.style.format({
    "price_month":     "{:,.0f} ₽",
    "price_sqm_month": "{:,.0f} ₽/м²",
    "area_sqm":        "{:.1f} м²",
}))

,url,price_month,area_sqm,price_sqm_month,metro_station,metro_walk_min,floor,floors_total
1,https://www.cian.ru/rent/commercial/327021596/,"592,920 ₽",648.0 м²,915 ₽/м²,None,nan,2,2
2,https://www.cian.ru/rent/commercial/326649120/,"1,100,928 ₽",1203.2 м²,915 ₽/м²,None,nan,2,2
3,https://www.cian.ru/rent/commercial/329159983/,"92,316 ₽",100.8 м²,916 ₽/м²,Калужская,11.000000,-1,6
4,https://www.cian.ru/rent/commercial/324910226/,"927,667 ₽",1012.0 м²,917 ₽/м²,None,nan,1,1
5,https://www.cian.ru/rent/commercial/324910984/,"1,871,834 ₽",2042.0 м²,917 ₽/м²,None,nan,1,1
6,https://www.cian.ru/rent/commercial/327175759/,"418,000 ₽",440.0 м²,950 ₽/м²,Дегунино,10.000000,-1,1
7,https://www.cian.ru/rent/commercial/309142954/,"105,900 ₽",110.0 м²,963 ₽/м²,None,nan,1,1
8,https://www.cian.ru/rent/commercial/276669208/,"28,900 ₽",30.0 м²,963 ₽/м²,None,nan,1,1
9,https://www.cian.ru/rent/commercial/329893622/,"50,000 ₽",51.9 м²,963 ₽/м²,Печатники,13.000000,-1,6
10,https://www.cian.ru/rent/commercial/278484632/,"87,000 ₽",90.0 м²,967 ₽/м²,None,nan,1,1


---
# Часть 7. Визуализация

Набор интерактивных графиков Plotly для изучения рынка коммерческой аренды в разрезе  
округов, типов объектов и ценовых категорий.


In [18]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px

# ── 1. Медиана цены по округам ────────────────────────────────────────────────
bar_df = (
    df.groupby("okrug")["price_sqm_month"]
    .median().sort_values(ascending=False).reset_index()
)
fig1 = px.bar(
    bar_df, x="okrug", y="price_sqm_month",
    title="Медиана цены аренды по округам, руб/м²/мес",
    labels={"okrug":"Округ","price_sqm_month":"руб/м²/мес"},
    color="price_sqm_month", color_continuous_scale="Blues",
    height=400,
)
fig1.update_layout(plot_bgcolor="white", coloraxis_showscale=False)
fig1.show()

# ── 2. Типы объектов + классы зданий ─────────────────────────────────────────
fig2 = px.pie(
    df["object_type"].value_counts().reset_index(),
    names="object_type", values="count",
    title="Структура по типам объектов",
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Pastel,
    height=380,
)
fig2.show()

# ── 3. Scatter: площадь vs цена за м² ────────────────────────────────────────
fig3 = px.scatter(
    df, x="area_sqm", y="price_sqm_month",
    color="object_type",
    size="price_month", size_max=18,
    hover_data=["okrug","district","metro_station","metro_walk_min","floor","address"],
    title="Площадь vs Цена за м²",
    labels={"area_sqm":"Площадь, м²","price_sqm_month":"руб/м²/мес","object_type":"Тип"},
    color_discrete_sequence=px.colors.qualitative.Set2,
    height=450,
    opacity=0.7,
)
fig3.update_layout(plot_bgcolor="white")
fig3.show()

# ── 4. Box: распределение цен по типам ───────────────────────────────────────
fig4 = px.box(
    df, x="object_type", y="price_sqm_month",
    color="object_type",
    title="Разброс цен по типам объектов",
    labels={"object_type":"Тип","price_sqm_month":"руб/м²/мес"},
    color_discrete_sequence=px.colors.qualitative.Set2,
    height=400,
)
fig4.update_layout(plot_bgcolor="white", showlegend=False)
fig4.show()

# ── 5. Bar: медиана цены по расстоянию до метро ───────────────────────────────
metro_df = (
    df[df["metro_walk_min"].notna()]
    .groupby("metro_tier", observed=True)["price_sqm_month"]
    .median().reset_index()
)
fig5 = px.bar(
    metro_df, x="metro_tier", y="price_sqm_month",
    title="Цена аренды vs расстояние до метро",
    labels={"metro_tier":"До метро","price_sqm_month":"руб/м²/мес"},
    color="price_sqm_month", color_continuous_scale="Oranges",
    height=380, text_auto=".0f",
)
fig5.update_layout(plot_bgcolor="white", coloraxis_showscale=False)
fig5.show()

# ── 6. Топ-20 по цене за м² ───────────────────────────────────────────────────
top_df = (
    df.sort_values("price_sqm_month", ascending=False)
    .head(20)[["object_type","okrug","district","address",
               "area_sqm","price_month","price_sqm_month","metro_station","metro_walk_min"]]
    .reset_index(drop=True)
)
top_df.index += 1
display(
    top_df.style.format({
        "price_month":     "{:,.0f} ₽",
        "price_sqm_month": "{:,.0f} ₽/м²",
        "area_sqm":        "{:.1f} м²",
    }).background_gradient(subset=["price_sqm_month"], cmap="YlOrRd")
)

,object_type,okrug,district,address,area_sqm,price_month,price_sqm_month,metro_station,metro_walk_min
1,Торговая площадь,ЦАО,Арбат,"Карманицкий, 9",4.1 м²,"155,000 ₽","37,805 ₽/м²",Смоленская,1.000000
2,Торговая площадь,ЦАО,Арбат,"Карманицкий, 9",4.0 м²,"140,000 ₽","35,000 ₽/м²",Смоленская,1.000000
3,Свободное назначение,ЦАО,Таганский,"Солянка, 2/6",23.7 м²,"720,000 ₽","30,380 ₽/м²",Китай-город,2.000000
4,Свободное назначение,ЦАО,Арбат,"Карманицкий, 9",26.3 м²,"740,000 ₽","28,137 ₽/м²",Смоленская,1.000000
5,Торговая площадь,ЦАО,Арбат,"Карманицкий, 9",22.3 м²,"620,000 ₽","27,803 ₽/м²",Смоленская,1.000000
6,Торговая площадь,ЦАО,Таганский,"Солянка, 2/6",58.3 м²,"1,500,000 ₽","25,729 ₽/м²",Китай-город,2.000000
7,Склад,ЦАО,Басманный,"Маросейка, 4/2С1",80.0 м²,"1,700,000 ₽","21,250 ₽/м²",Китай-город,2.000000
8,Торговая площадь,САО,Беговой,"Ленинградский, 33К3",15.3 м²,"310,000 ₽","20,261 ₽/м²",Динамо,3.000000
9,Свободное назначение,САО,Аэропорт,"Ленинградский, 62",10.1 м²,"185,000 ₽","18,317 ₽/м²",Аэропорт,2.000000
10,Торговая площадь,САО,Беговой,"Ленинградский, 4/2",18.1 м²,"300,000 ₽","16,575 ₽/м²",Белорусская,2.000000


---
# Часть 8. Выводы для DarkSpot

## 8.1. Что собрали

- **Источник:** ЦИАН - коммерческая аренда Москвы, статический парсинг через cloudscraper
- **Объём:** офисы, торговые площади, свободное назначение, склады
- **Поля:** цена аренды, площадь, тип объекта, округ, район, метро, этаж, класс здания, ремонт

## 8.2. Ключевые находки

### Цены по округам
- **ЦАО** стабильно лидирует по цене аренды - ставки в 1.5-2 раза выше, чем в периферийных округах
- **ЗАО и ЮЗАО** занимают второй ценовой эшелон - хорошая транспортная доступность при умеренных ставках
- **ЮВАО и ЮАО** - самые доступные округа, актуальны для форматов с высокой чувствительностью к аренде (ПВЗ, склады)

### Влияние метро
- Помещения в радиусе 5 минут пешком от метро стоят дороже - близость к метро является значимым ценообразующим фактором
- Для форматов с высоким пешеходным трафиком (кофейня, торговля) близость к метро критична
- Для складов и офисов back-office удалённость от метро менее критична и позволяет сэкономить

### Структура предложения
- Наибольший объём предложения - офисы и торговые площади
- Эконом-сегмент (<5 000 руб/м²/мес) преобладает в складском и офисном сегментах
- Люкс-сегмент (>40 000 руб/м²/мес) сосредоточен в ЦАО, объекты стрит-ритейла

### Площадь и цена
- Обратная зависимость: чем больше площадь - тем ниже ставка за м²
- Малые форматы 20-80 м² имеют наибольшую удельную цену - наиболее ликвидный сегмент
- Крупные площади >500 м² дают дисконт, но требуют большего капитала на запуск

## 8.3. Рекомендации для DarkSpot

| Формат бизнеса | Рекомендованный округ | Оптимальная площадь | Обоснование |
|---|---|---|---|
| Кофейня | ЦАО, ЗАО | 30-60 м² | Высокий пешеходный трафик, стрит-ритейл |
| Салон красоты | ЮЗАО, ЗАО | 40-80 м² | Платёжеспособная аудитория, умеренная аренда |
| ПВЗ / Пункт выдачи | ЮВАО, ЮАО | 20-40 м² | Плотное население, минимальные ставки |
| Офис | САО, СВАО | 50-150 м² | Баланс цены и транспортной доступности |

## 8.4. Синергия с блоком ЦА

Данные по аренде (этот блок) в сочетании с данными по целевой аудитории (блок 1) дают  
комплексный сигнал для DarkSpot:

**Высокая плотность платёжеспособной аудитории + разумная арендная ставка = приоритетная локация**

Итоговый скоринг локации должен учитывать оба параметра одновременно, а не каждый по отдельности.
